# Movies Project Starter

Use this starter notebook for the BAN 6003 final project option: **The Movies Dataset / MovieLens ratings**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head())


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,0,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02
3,1,1129,2.0,1260759185,2009-12-14 02:53:05
4,1,1172,4.0,1260759205,2009-12-14 02:53:25



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret
3,31357,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine
4,11862,Charles Shyer,Steve Martin|Diane Keaton|Martin Short



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men
3,31357,based on novel|interracial relationship|single...
4,11862,baby|midlife crisis|confidence|aging|daughter|...


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


Markdown 1: We are working with 5 different folders in this project. We seem to be looking at information regarding 9,082 movies, but the ratings and links folders contain more than 9,082 columns (movies). The summary also shows that there are no duplicate rows, but that 4 out of the 5 folders contain missing cells. Keywords has the most amount of missing cells (761), and ratings has the least amount of missing cells (0).

### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [5]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


Markdown 2: None of the folders seem to contain duplicate keys, nor missing key values.

## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question: Which movie characteristics appear most useful for content strategy?

- Why this question matters: Customers will only subscribe or remain subscribed to streaming platforms if they guarantee them a source of entertainment. Different streaming services are competing to see which one has the most appealing catalog that provides that entertainment. Defining what appealing means will be the goal of this project.

- Tables/files used: The files used for this project are: movies, ratings, credits and keywords.

- Row meaning of each important table: 
    - Movies table: Each row represents a different movie
    - Ratings: each rating represents a user's rating of a particular movie. A user can review more than one movie, hence why there are a 99,810 rows.
    - Credits: each row contains a movie's director and top cast members.
    - Keywords: each row assigns a set of words that identify a movie based on their tmbdId.

- Early data quality issues: For the movie table, the budget and revenue column contains very large integer entries that are unclear, additionally some entries contain zeros, which could imply that some of these zero entries are false. They need to be investigated later. For popularity it gives numbers but it is difficult to understand what they mean, due to them having a lot of decimal places behind those numbers. The adult column seems to always be false regardless of the genre. The credits and keywords tables have missing values that need to be checked.

- Possible target/outcome: We can look at the average rating score and compare movie scores to classify them as more appealing or less appealing. Additionally, we can look into keywords, top cast, and directors to see if having a specific cast and/or director is a factor that is associated with higher audience ratings.

- Ethics, privacy, or governance concerns: The ratings table's userId can be classified as a confidential internal identifier, so it could potentially identify an individual. I should also document where the data I will be using comes from and how I am going to relate the tables to one another. For instance, I can relate the movies table and the ratings table using movieId and see if there is a relationship between ratings and columns such as revenue, budget, genre, and others. I will not use an individual user's rating as the primary measure of success. Instead, I will use ratings in an aggregated way and compare them with different movie characteristics to reach a better understanding of audience response.-



In [6]:
# Milestone 1 code workspace
# Add your additional profiling checks here.
# Examples: missingness by important columns, duplicate key checks, suspicious value checks.

First it is important to show the basic structure of the movies, credits and keywords DataFrames.

In [7]:
print(movies.shape)
num_rows, num_columns = movies.shape
print(f"This dataset has {num_rows} rows and {num_columns} columns.")
df.columns
print(movies.head()) 
print(movies.tail())

(9082, 21)
This dataset has 9082 rows and 21 columns.
   movieId  tmdbId    imdb_id                        title  \
0        1     862  tt0114709                    Toy Story   
1        2    8844  tt0113497                      Jumanji   
2        3   15602  tt0113228             Grumpier Old Men   
3        4   31357  tt0114885            Waiting to Exhale   
4        5   11862  tt0113041  Father of the Bride Part II   

                original_title original_language release_date  runtime  \
0                    Toy Story                en   1995-10-30     81.0   
1                      Jumanji                en   1995-12-15    104.0   
2             Grumpier Old Men                en   1995-12-22    101.0   
3            Waiting to Exhale                en   1995-12-22    127.0   
4  Father of the Bride Part II                en   1995-02-10    106.0   

     budget      revenue  popularity  vote_average  vote_count  \
0  30000000  373554033.0   21.946943           7.7      5415.0

In [8]:
print(credits.shape)
num_rows, num_columns = credits.shape
print(f"This dataset has {num_rows} rows and {num_columns} columns.")
df.columns
print(credits.head()) 
print(credits.tail())

(9082, 3)
This dataset has 9082 rows and 3 columns.
   tmdbId         director                                       top_cast
0     862    John Lasseter                Tom Hanks|Tim Allen|Don Rickles
1    8844     Joe Johnston     Robin Williams|Jonathan Hyde|Kirsten Dunst
2   15602    Howard Deutch         Walter Matthau|Jack Lemmon|Ann-Margret
3   31357  Forest Whitaker  Whitney Houston|Angela Bassett|Loretta Devine
4   11862    Charles Shyer         Steve Martin|Diane Keaton|Martin Short
      tmdbId            director  \
9077  159550      Gregg Champion   
9078  392572   Tinu Suresh Desai   
9079  402672  Ashutosh Gowariker   
9080  315011        Hideaki Anno   
9081  391698          Ron Howard   

                                               top_cast  
9077        Sidney Poitier|Wendy Crewson|Jay O. Sanders  
9078              Akshay Kumar|Ileana D'Cruz|Esha Gupta  
9079              Hrithik Roshan|Pooja Hegde|Kabir Bedi  
9080  Hiroki Hasegawa|Yutaka Takenouchi|Satomi Ishihara

In [9]:
print(keywords.shape)
num_rows, num_columns = keywords.shape
print(f"This dataset has {num_rows} rows and {num_columns} columns.")
df.columns
print(keywords.head()) 
print(keywords.tail())

(9082, 2)
This dataset has 9082 rows and 2 columns.
   tmdbId                                      keywords_list
0     862  jealousy|toy|boy|friendship|friends|rivalry|bo...
1    8844  board game|disappearance|based on children's b...
2   15602   fishing|best friend|duringcreditsstinger|old men
3   31357  based on novel|interracial relationship|single...
4   11862  baby|midlife crisis|confidence|aging|daughter|...
      tmdbId                                      keywords_list
9077  159550                            friendship|brick making
9078  392572                                          bollywood
9079  402672                                          bollywood
9080  315011  monster|godzilla|giant monster|destruction|kai...
9081  391698                                  music|documentary


- Initial cleaning plan: My plan would be:
1. For the first step, I will inspect the large integer entries using movies['column'].describe(). 

In [10]:
movies[['budget', 'revenue']].describe()

,budget,revenue
count,9.082000e+03,9.082000e+03
mean,1.664789e+07,4.918728e+07
std,3.345276e+07,1.301801e+08
min,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00
50%,1.000000e+05,2.716080e+05
75%,2.000000e+07,3.672528e+07
max,3.800000e+08,2.787965e+09


2. Count the zero values for both columns using (movies[['col1', 'col2']] == 0).sum() . These values need to be investigated later, because some of them could be false zeros.

In [11]:
#Count of zero values
(movies[['budget', 'revenue']] == 0).sum()

budget     4443
revenue    4322
dtype: int64

3. Then, finding the mean using ['column'].describe() to print the count, mean, standard deviation, 50% minimum and maximum values.

In [12]:
movies[['popularity']].describe()

,popularity
count,9082.000000
mean,7.268905
std,9.372314
min,0.000004
25%,2.954521
50%,6.418871
75%,9.861742
max,547.488298


In [13]:
# Example: summary statistics for age
print("Count:", movies["popularity"].count())
print("Mean:", movies["popularity"].mean())
print("Median:", movies["popularity"].median())
print("Minimum:", movies["popularity"].min())
print("Maximum:", movies["popularity"].max())
print("Standard deviation:", movies["popularity"].std())

Count: 9082
Mean: 7.2689049122439995
Median: 6.418870500000001
Minimum: 4e-06
Maximum: 547.488298
Standard deviation: 9.372314028735433


4. For the adult column I am going to inspect data types, non-missing counts, and value counts.

In [14]:
movies.info()
movies["adult"].value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 9082 entries, 0 to 9081
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   movieId               9082 non-null   int64  
 1   tmdbId                9082 non-null   int64  
 2   imdb_id               9082 non-null   str    
 3   title                 9082 non-null   str    
 4   original_title        9082 non-null   str    
 5   original_language     9082 non-null   str    
 6   release_date          9082 non-null   str    
 7   runtime               9082 non-null   float64
 8   budget                9082 non-null   int64  
 9   revenue               9082 non-null   float64
 10  popularity            9082 non-null   float64
 11  vote_average          9082 non-null   float64
 12  vote_count            9082 non-null   float64
 13  genres                9082 non-null   str    
 14  production_companies  9082 non-null   str    
 15  production_countries  9082 non-n

adult
False    9082
Name: count, dtype: int64

5. Lastly, I am going to inspect the missing counts for the credits and keynotes columns. 

In [15]:
credits.info()
credits.dtypes
print(credits.isna().sum())
credits["director"].value_counts()
credits["top_cast"].value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 9082 entries, 0 to 9081
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   tmdbId    9082 non-null   int64
 1   director  9058 non-null   str  
 2   top_cast  8992 non-null   str  
dtypes: int64(1), str(2)
memory usage: 213.0 KB
tmdbId       0
director    24
top_cast    90
dtype: int64


top_cast
Daniel Radcliffe|Rupert Grint|Emma Watson            7
William Shatner|Leonard Nimoy|DeForest Kelley        6
Groucho Marx|Chico Marx|Harpo Marx                   6
Louis C.K.                                           6
Groucho Marx|Harpo Marx|Chico Marx                   5
                                                    ..
Sidney Poitier|Wendy Crewson|Jay O. Sanders          1
Akshay Kumar|Ileana D'Cruz|Esha Gupta                1
Hrithik Roshan|Pooja Hegde|Kabir Bedi                1
Hiroki Hasegawa|Yutaka Takenouchi|Satomi Ishihara    1
Paul McCartney|Ringo Starr|John Lennon               1
Name: count, Length: 8837, dtype: int64

In [16]:
keywords.info()
keywords.dtypes
print(keywords.isna().sum())
keywords["keywords_list"].value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 9082 entries, 0 to 9081
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   tmdbId         9082 non-null   int64
 1   keywords_list  8321 non-null   str  
dtypes: int64(1), str(1)
memory usage: 142.0 KB
tmdbId             0
keywords_list    761
dtype: int64


keywords_list
independent film                                          92
woman director                                            62
musical                                                   20
duringcreditsstinger                                      19
sport                                                     17
                                                          ..
mansion                                                    1
shark attack|sequel|farce|las vegas|creature feature       1
friendship|brick making                                    1
monster|godzilla|giant monster|destruction|kaiju|toyko     1
music|documentary                                          1
Name: count, Length: 7951, dtype: int64

In [17]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,str
title,str
original_title,str
original_language,str
release_date,str
runtime,float64
budget,int64
revenue,float64


,missing_rate
genres_list,0.003854
primary_genre,0.003854
status,0.000220
tmdbId,0.000000
movieId,0.000000
original_title,0.000000
title,0.000000
imdb_id,0.000000
original_language,0.000000
revenue,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[us]


,missing_rate
userId,0.0
movieId,0.0
rating,0.0
timestamp,0.0
rating_datetime,0.0


... (5 outputs left out)

,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,str


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



## Updated Project Direction

Depending on the number of ratings, average audience rating can help determine whether a movie appeals to general audiences. Although a high average rating does not necessarily indicate commercial success, it provides a useful measure of audience response. By comparing average ratings across different movie characteristics, we can identify patterns that may help us better understand what audiences look for in a movie.

In [18]:
print("Movie Columns:")
print(movies.columns.tolist())

print("Rating Columns:")
print(ratings.columns.tolist())

print("Credits Columns:")
print(credits.columns.tolist())

print("Keywords Columns:")
print(keywords.columns.tolist())

Movie Columns:
['movieId', 'tmdbId', 'imdb_id', 'title', 'original_title', 'original_language', 'release_date', 'runtime', 'budget', 'revenue', 'popularity', 'vote_average', 'vote_count', 'genres', 'production_companies', 'production_countries', 'adult', 'status', 'genres_list', 'primary_genre', 'release_year']
Rating Columns:
['userId', 'movieId', 'rating', 'timestamp', 'rating_datetime']
Credits Columns:
['tmdbId', 'director', 'top_cast']
Keywords Columns:
['tmdbId', 'keywords_list']


In [19]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.
import sqlite3
import duckdb
import pandas as pd
from pathlib import Path

# 1. Define project directory
repo_root = Path.cwd().parent

# 2. Create SQLite database
output_dir = repo_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

db_path = output_dir / "movies.db"
sqlite_connection = sqlite3.connect(db_path)

# 3. Load DataFrames into SQLite
movies.to_sql("movies", sqlite_connection, if_exists="replace", index=False)
ratings.to_sql("ratings", sqlite_connection, if_exists="replace", index=False)
credits.to_sql("credits", sqlite_connection, if_exists="replace", index=False)
keywords.to_sql("keywords", sqlite_connection, if_exists="replace", index=False)

# 4. Create DuckDB connection
duckdb_connection = duckdb.connect()

In [20]:
rating_summary = (
    ratings.groupby("movieId")
    .agg(
        user_rating_mean=("rating", "mean"),
        user_rating_count=("rating", "size"),
        user_rating_std=("rating", "std"),
        first_rating_date=("rating_datetime", "min"),
        last_rating_date=("rating_datetime", "max")
    )
    .reset_index()
)

abt = (
    movies
    .merge(rating_summary, on="movieId", how="left", validate="one_to_one")
    .merge(credits, on="tmdbId", how="left", validate="one_to_one")
    .merge(keywords, on="tmdbId", how="left", validate="one_to_one")
)

abt["has_revenue"] = (pd.to_numeric(abt["revenue"], errors="coerce") > 0).astype("Int64")
abt["rating_target_eligible"] = abt["user_rating_count"].ge(20)
abt["high_user_rating"] = (
    abt["user_rating_mean"].ge(3.75)
    .where(abt["rating_target_eligible"])
    .astype("Int64")
)

## Cleaning and Transformation Decisions

### Movies Cleaning and Transformation

In [21]:
missing_movies = movies.isna().sum().sort_values(ascending=False)

print("Missing values by column:")
display(missing_movies[missing_movies > 0])

Missing values by column:


genres_list      35
primary_genre    35
status            2
dtype: int64

In [22]:
print("Duplicate movieId values:", movies["movieId"].duplicated().sum())

print("Duplicate tmdbId values:", movies["tmdbId"].duplicated().sum())

Duplicate movieId values: 0
Duplicate tmdbId values: 0


In [23]:
display(
    movies[["runtime", "budget", "revenue", "popularity"]].describe()
)

,runtime,budget,revenue,popularity
count,9082.000000,9.082000e+03,9.082000e+03,9082.000000
mean,105.661418,1.664789e+07,4.918728e+07,7.268905
std,30.350712,3.345276e+07,1.301801e+08,9.372314
min,0.000000,0.000000e+00,0.000000e+00,0.000004
25%,93.000000,0.000000e+00,0.000000e+00,2.954521
50%,102.000000,1.000000e+05,2.716080e+05,6.418871
75%,115.000000,2.000000e+07,3.672528e+07,9.861742
max,1140.000000,3.800000e+08,2.787965e+09,547.488298


In [24]:
print("Zero runtimes:", (movies["runtime"] == 0).sum())
print("Zero budgets:", (movies["budget"] == 0).sum())
print("Zero revenues:", (movies["revenue"] == 0).sum())

Zero runtimes: 20
Zero budgets: 4443
Zero revenues: 4322


In [25]:
display(
    movies.loc[
        (movies["runtime"] == 0) |
        (movies["budget"] == 0) |
        (movies["revenue"] == 0),
        ["movieId", "title", "runtime", "budget", "revenue"]
    ].head(20)
)

,movieId,title,runtime,budget,revenue
2,3,Grumpier Old Men,101.0,0,0.0
4,5,Father of the Bride Part II,106.0,0,76578911.0
6,7,Sabrina,127.0,58000000,0.0
7,8,Tom and Huck,97.0,0,0.0
11,12,Dracula: Dead and Loving It,88.0,0,0.0
12,13,Balto,78.0,0,11348324.0
21,22,Copycat,124.0,0,0.0
23,24,Powder,111.0,0,0.0
25,26,Othello,123.0,0,0.0
27,28,Persuasion,104.0,0,0.0


In [26]:
# Replace zero values with NaN
movies[["runtime", "budget", "revenue"]] = (
    movies[["runtime", "budget", "revenue"]].replace(0, np.nan)
)

In [27]:
print("Remaining zero values:")

print("Runtime:", (movies["runtime"] == 0).sum())
print("Budget:", (movies["budget"] == 0).sum())
print("Revenue:", (movies["revenue"] == 0).sum())

print("\nUpdated missing values:")

display(movies[["runtime", "budget", "revenue"]].isna().sum())

Remaining zero values:
Runtime: 0
Budget: 0
Revenue: 0

Updated missing values:


runtime      20
budget     4443
revenue    4322
dtype: int64

In [28]:
movies["release_decade"] = (movies["release_year"] // 10) * 10

display(movies["release_decade"].value_counts().sort_index())

release_decade
1900       1
1910       6
1920      56
1930     148
1940     250
1950     341
1960     443
1970     549
1980    1193
1990    2193
2000    2533
2010    1369
Name: count, dtype: int64

In [29]:
movies["Short"] = movies["runtime"] < 90
movies["Normal"] = (movies["runtime"] >= 90) & (movies["runtime"] <=120)
movies["Long"] = movies["runtime"] > 120

display(movies[movies["Short"] == True])
display(movies[movies["Normal"] == True])
display(movies[movies["Long"] == True])


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,release_decade,Short,Normal,Long
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000.0,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995,1990,True,False,False
11,12,12110,tt0112896,Dracula: Dead and Loving It,Dracula: Dead and Loving It,en,1995-12-22,88.0,NaN,NaN,5.430331,5.7,210.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam...","[{'name': 'Columbia Pictures', 'id': 5}, {'nam...","[{'iso_3166_1': 'FR', 'name': 'France'}, {'iso...",False,Released,Comedy|Horror,Comedy,1995,1990,True,False,False
12,13,21032,tt0112453,Balto,Balto,en,1995-12-22,78.0,NaN,11348324.0,12.140733,7.1,423.0,"[{'id': 10751, 'name': 'Family'}, {'id': 16, '...","[{'name': 'Universal Pictures', 'id': 33}, {'n...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Family|Animation|Adventure,Family,1995,1990,True,False,False
32,34,9598,tt0112431,Babe,Babe,en,1995-07-18,89.0,30000000.0,254134910.0,14.404764,6.0,756.0,"[{'id': 14, 'name': 'Fantasy'}, {'id': 18, 'na...","[{'name': 'Universal Pictures', 'id': 33}, {'n...","[{'iso_3166_1': 'AU', 'name': 'Australia'}, {'...",False,Released,Fantasy|Drama|Comedy|Family,Fantasy,1995,1990,True,False,False
35,37,139405,tt0112286,Across the Sea of Time,Across the Sea of Time,en,1995-10-20,51.0,NaN,NaN,0.114469,3.5,2.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 36, '...","[{'name': 'Sony New Technologies', 'id': 53390}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|History|Drama|Family,Adventure,1995,1990,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9070,161084,368620,tt4465220,My Friend Rockefeller,My Friend Rockefeller,en,2015-07-04,87.0,NaN,NaN,0.614048,6.2,9.0,"[{'id': 99, 'name': 'Documentary'}]",[],[],False,Released,Documentary,Documentary,2015,2010,True,False,False
9071,161155,401387,tt5794766,Sunspring,Sunspring,en,2016-06-09,9.0,NaN,NaN,1.124787,6.6,7.0,"[{'id': 878, 'name': 'Science Fiction'}]","[{'name': 'End Cue', 'id': 65906}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Science Fiction,Science Fiction,2016,2010,True,False,False
9075,161830,314420,tt3732950,Body,Body,en,2015-01-25,75.0,NaN,NaN,6.236714,5.0,16.0,"[{'id': 18, 'name': 'Drama'}, {'id': 53, 'name...","[{'name': 'Oscilloscope Laboratories', 'id': 4...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Drama|Thriller,Drama,2015,2010,True,False,False
9076,161918,390989,tt4831420,Sharknado 4: The 4th Awakens,Sharknado 4: The 4th Awakens,en,2016-07-31,85.0,NaN,NaN,4.574494,4.3,88.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 27, 'nam...","[{'name': 'The Asylum', 'id': 1311}, {'name': ...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Horror|Science Fiction,Comedy,2016,2010,True,False,False


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,release_decade,Short,Normal,Long
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000.0,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995,1990,False,True,False
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,NaN,NaN,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,1990,False,True,False
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,NaN,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995,1990,False,True,False
7,8,45325,tt0112302,Tom and Huck,Tom and Huck,en,1995-12-22,97.0,NaN,NaN,2.561161,5.4,45.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Walt Disney Pictures', 'id': 2}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Drama|Family,Action,1995,1990,False,True,False
8,9,9091,tt0114576,Sudden Death,Sudden Death,en,1995-12-22,106.0,35000000.0,64350171.0,5.231580,5.5,174.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Universal Pictures', 'id': 33}, {'n...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Thriller,Action,1995,1990,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9072,161336,373348,tt5278462,Author: The JT LeRoy Story,Author: The JT LeRoy Story,en,2016-01-22,110.0,NaN,NaN,0.428899,6.2,9.0,"[{'id': 99, 'name': 'Documentary'}]",[],[],False,Released,Documentary,Documentary,2016,2010,False,True,False
9073,161582,338766,tt2582782,Hell or High Water,Hell or High Water,en,2016-08-12,102.0,12000000.0,37589296.0,12.565896,7.2,1304.0,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...","[{'name': 'Sidney Kimmel Entertainment', 'id':...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Crime|Drama|Thriller|Western,Crime,2016,2010,False,True,False
9074,161594,390734,tt5595168,Kingsglaive: Final Fantasy XV,キングスグレイブ ファイナルファンタジーXV,ja,2016-07-09,115.0,NaN,269980.0,7.692445,6.8,201.0,"[{'id': 28, 'name': 'Action'}, {'id': 16, 'nam...","[{'name': 'Square Enix', 'id': 10038}, {'name'...","[{'iso_3166_1': 'JP', 'name': 'Japan'}, {'iso_...",False,Released,Action|Animation|Adventure|Drama|Fantasy,Action,2016,2010,False,True,False
9080,163056,315011,tt4262980,Shin Godzilla,シン・ゴジラ,ja,2016-07-29,120.0,15000000.0,77000000.0,9.285519,6.6,152.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Cine Bazar', 'id': 5896}, {'name': ...","[{'iso_3166_1': 'JP', 'name': 'Japan'}]",False,Released,Action|Adventure|Drama|Horror|Science Fiction,Action,2016,2010,False,True,False


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,release_decade,Short,Normal,Long
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000.0,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995,1990,False,False,True
5,6,949,tt0113277,Heat,Heat,en,1995-12-15,170.0,60000000.0,187436818.0,17.924927,7.7,1886.0,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...","[{'name': 'Regency Enterprises', 'id': 508}, {...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Crime|Drama|Thriller,Action,1995,1990,False,False,True
6,7,11860,tt0114319,Sabrina,Sabrina,en,1995-12-15,127.0,58000000.0,NaN,6.677277,6.2,141.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '...","[{'name': 'Paramount Pictures', 'id': 4}, {'na...","[{'iso_3166_1': 'DE', 'name': 'Germany'}, {'is...",False,Released,Comedy|Romance,Comedy,1995,1990,False,False,True
9,10,710,tt0113189,GoldenEye,GoldenEye,en,1995-11-16,130.0,58000000.0,352194034.0,14.686036,6.6,1194.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...","[{'name': 'United Artists', 'id': 60}, {'name'...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...",False,Released,Adventure|Action|Thriller,Adventure,1995,1990,False,False,True
13,14,10858,tt0113987,Nixon,Nixon,en,1995-12-22,192.0,44000000.0,13681765.0,5.092000,7.1,72.0,"[{'id': 36, 'name': 'History'}, {'id': 18, 'na...","[{'name': 'Hollywood Pictures', 'id': 915}, {'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,History|Drama,History,1995,1990,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9053,159093,291805,tt3110958,Now You See Me 2,Now You See Me 2,en,2016-06-02,129.0,90000000.0,334901337.0,39.540653,6.7,3362.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Summit Entertainment', 'id': 491}, ...","[{'iso_3166_1': 'CA', 'name': 'Canada'}, {'iso...",False,Released,Action|Adventure|Comedy|Crime|Mystery,Action,2016,2010,False,False,True
9057,159858,259693,tt3065204,The Conjuring 2,The Conjuring 2,en,2016-05-13,134.0,40000000.0,320170008.0,14.767317,7.0,2018.0,"[{'id': 27, 'name': 'Horror'}]","[{'name': 'New Line Cinema', 'id': 12}, {'name...","[{'iso_3166_1': 'CA', 'name': 'Canada'}, {'iso...",False,Released,Horror,Horror,2016,2010,False,False,True
9061,160438,324668,tt4196776,Jason Bourne,Jason Bourne,en,2016-07-27,123.0,120000000.0,415484914.0,19.133256,5.9,2386.0,"[{'id': 28, 'name': 'Action'}, {'id': 53, 'nam...","[{'name': 'The Kennedy/Marshall Company', 'id'...","[{'iso_3166_1': 'GB', 'name': 'United Kingdom'...",False,Released,Action|Thriller,Action,2016,2010,False,False,True
9078,162542,392572,tt5165344,Rustom,रुस्तम,hi,2016-08-12,150.0,1000000.0,NaN,7.333139,7.3,25.0,"[{'id': 53, 'name': 'Thriller'}, {'id': 10749,...","[{'name': 'KriArj Entertainment', 'id': 91689}]","[{'iso_3166_1': 'IN', 'name': 'India'}]",False,Released,Thriller|Romance,Thriller,2016,2010,False,False,True


### Ratings Cleaning and Transformation

In [30]:
missing_ratings = ratings.isna().sum().sort_values(ascending=False)

print("Missing values by column:")
display(missing_ratings)

Missing values by column:


userId             0
movieId            0
rating             0
timestamp          0
rating_datetime    0
dtype: int64

In [31]:
print("Complete duplicate rows:",
      ratings.duplicated().sum())

print("Duplicate userId/movieId combinations:",
      ratings.duplicated(subset=["userId", "movieId"]).sum())

Complete duplicate rows: 0
Duplicate userId/movieId combinations: 0


In [47]:
display(ratings["rating"].describe())

print("Unique rating values:")
display(sorted(ratings["rating"].dropna().unique()))

count    99810.000000
mean         3.542972
std          1.057939
min          0.500000
25%          3.000000
50%          4.000000
75%          4.000000
max          5.000000
Name: rating, dtype: float64

Unique rating values:


[np.float64(0.5),
 np.float64(1.0),
 np.float64(1.5),
 np.float64(2.0),
 np.float64(2.5),
 np.float64(3.0),
 np.float64(3.5),
 np.float64(4.0),
 np.float64(4.5),
 np.float64(5.0)]

In [63]:
display(ratings["rating"].value_counts().sort_index())

rating
0.5     1097
1.0     3322
1.5     1685
2.0     7263
2.5     4446
3.0    20039
3.5    10512
4.0    28705
4.5     7698
5.0    15043
Name: count, dtype: int64

### Credits Cleaning and Transformation

In [79]:
missing_credits = credits.isna().sum().sort_values(ascending=False)

print("Missing values in Credits:")
display(missing_credits[missing_credits > 0])

Missing values in Credits:


top_cast    90
director    24
dtype: int64

In [92]:
print("Duplicate tmdbId values:",
      credits["tmdbId"].duplicated().sum())

Duplicate tmdbId values: 0


### Keywords Cleaning and Transformation

In [118]:
missing_keywords = keywords.isna().sum().sort_values(ascending=False)

print("Missing values in Keywords:")
display(missing_keywords[missing_keywords > 0])

Missing values in Keywords:


keywords_list    761
dtype: int64

In [128]:
print("Duplicate tmdbId values:",
      keywords["tmdbId"].duplicated().sum())

Duplicate tmdbId values: 0


In [138]:
print("Movies with keywords:", keywords["keywords_list"].notna().sum())
print("Movies without keywords:", keywords["keywords_list"].isna().sum())

Movies with keywords: 8321
Movies without keywords: 761


I started by reviewing all four datasets (Movies, Ratings, Credits, and Keywords) to understand their structure and identify any data quality issues. First we counted missing and duplictaed values in every table. 

For the `movies` table I displayed the summary statistics and I used them to count how many zero values there were for `runtimes`, `budgets`, and `revenues`. Once they were found, I replaced the 0 entries with NaN. Then I created a new column called release date to see how many movies were released each decade. Finally I created three new columns labeled `Short`, `Normal`, and `Long` to classify movies based on their runtime. 

For the `ratings` table I wanted to see the different ratings that movies have received, so I displayed the ratings which range from 0.5 to 1.0.

For the `credits` table I examined missing values in the director and cast columns. Since these are potential explanatory variables in our research question. 

For the `keywords` I checked whether a movie's TMDB identifier appears more than once because I want to confirm that the Keywords table has one record per movie before joining it to Movies.

### Aggregation and Row Meaning

In [148]:
audience_rating = (
    ratings
    .groupby("movieId")
    .agg(
        avg_rating=("rating", "mean"),
        rating_count=("rating", "count")
    )
    .reset_index()
)

display(audience_rating.head())

,movieId,avg_rating,rating_count
0,1,3.872470,247
1,2,3.401869,107
2,3,3.161017,59
3,4,2.384615,13
4,5,3.267857,56


In [157]:
display(
    audience_rating[["avg_rating", "rating_count"]].describe()
)

,avg_rating,rating_count
count,9025.000000,9025.000000
mean,3.290203,11.059280
std,0.881612,24.095433
min,0.500000,1.000000
25%,2.833333,1.000000
50%,3.500000,3.000000
75%,3.961538,9.000000
max,5.000000,341.000000


### Data Integration and SQL/Database Checks

In [158]:
movies_abt = duckdb_connection.execute("""
WITH audience_rating AS (
    SELECT
        movieId,
        AVG(rating) AS user_rating_mean,
        COUNT(rating) AS user_rating_count
    FROM ratings
    GROUP BY movieId
)
SELECT
    ar.movieId,
    ar.user_rating_mean,
    ar.user_rating_count,
    m.popularity,
    m.budget,
    m.revenue,
    m.tmdbId,
    m.original_title,
    c.director,
    c.top_cast,
    k.keywords_list
FROM audience_rating AS ar
LEFT JOIN movies AS m
    ON ar.movieId = m.movieId
LEFT JOIN credits AS c
    ON m.tmdbId = c.tmdbId
LEFT JOIN keywords AS k
    ON m.tmdbId = k.tmdbId
""").df()

movies_abt.head()

,movieId,user_rating_mean,user_rating_count,popularity,budget,revenue,tmdbId,original_title,director,top_cast,keywords_list
0,31,3.178571,42,9.481338,NaN,180000000.0,9909,Dangerous Minds,John N. Smith,Michelle Pfeiffer|George Dzundza|Courtney B. V...,rap music|karate|high school|naval officer|tea...
1,1061,3.545455,33,9.198283,44000000.0,165615285.0,819,Sleepers,Barry Levinson,Robert De Niro|Brad Pitt|Kevin Bacon,child abuse|sadistic|sexual abuse|pastor|repay...
2,1263,3.864583,48,7.729445,15000000.0,50000000.0,11778,The Deer Hunter,Michael Cimino,Robert De Niro|John Cazale|John Savage,vietnam veteran|pennsylvania|vietnam|party|esc...
3,1293,3.978261,46,12.383755,22000000.0,77737889.0,783,Gandhi,Richard Attenborough,Ben Kingsley|Candice Bergen|Edward Fox,muslim|indian lead|demonstration|world war ii|...
4,1343,3.743590,39,10.302735,35000000.0,182300000.0,1598,Cape Fear,Martin Scorsese,Robert De Niro|Nick Nolte|Jessica Lange,prison|child abuse|rape|small town|daughter|po...


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [159]:
data_dictionary = pd.DataFrame({
    "column_name": movies_abt.columns,
    "data_type": [str(movies_abt[column].dtype) for column in movies_abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

#Document description for MovieId
data_dictionary.loc[
    data_dictionary["column_name"] == "movieId",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = ["Unique identifier per movie.",
     "Movies dataset",
     "No transformation; retained from the original dataset.",
     "Used this as a key to join audience rating CTE to movies table."
]

#Document description for rating
data_dictionary.loc[
    data_dictionary["column_name"] == "rating",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = ["Movie rating score on a scale of 1.0 to 5.0.",
     "Ratings dataset",
     "No transformation; retained from the original dataset.",
     "If we want to predict the mean average ratings, ratings themselves cannot be used as predictors."
]

#Document description for User-Rating Mean
data_dictionary.loc[
    data_dictionary["column_name"] == "user_rating_mean",
    ["description", "source",  "transformation_or_derivation", "notes_or_concerns"]
    ] = ["Mean audience rating per movie.",
     "Ratings dataset",
     "Calculated using AVG(rating), grouped by movieId",
     "Used mean values of 100 or more as a parameter to filter rows. Rows are ordered by descending mean. "
]
   

#Document description for User-Rating Count
data_dictionary.loc[
    data_dictionary["column_name"] == "user_rating_count",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = ["Number of audience ratings per movie.", 
     "Ratings dataset",
     "Calculated using COUNT(rating), grouped by movieId",
     "Number of ratings per movie. Some movies have a small number or ratings."
]

#Document description for popularity
data_dictionary.loc[
    data_dictionary["column_name"] == "popularity",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = ["Popularity score of the movie.",
     "Movies dataset",
     "No transformation; retained from the original dataset.",
     "Popularity scores may differ based on the movie."
 ]

#Document description for budget
data_dictionary.loc[
    data_dictionary["column_name"] == "budget",
    ["description", "source","transformation_or_derivation", "notes_or_concerns"]
] = ["Reported production budget.",
    "Movies dataset",
    "No transformation; retained from the original dataset.",
    "Some of the budget entries contain zeros. This could be due to missing or unreported information."
]

#Document description for revenue
data_dictionary.loc[
    data_dictionary["column_name"] == "revenue",
    ["description", "source","transformation_or_derivation", "notes_or_concerns"]
] = ["Reported movie revenue.",
    "Movies dataset",
    "No transformation; retained from the original dataset.",
    "Some of the revenue entries contain zeros. This could be due to missing or unreported information."
] 

#Document description for tbdbId
data_dictionary.loc[
    data_dictionary["column_name"] == "tmdbId",
    ["description", "source","transformation_or_derivation", "notes_or_concerns"]
] = ["Unique identifier in The Movie Database",
    "Movies & Credits datasets",
    "No transformation; retained from the original dataset.",
    "Used to connect the Movies dataset to the Credits and Keywords datasets."
] 

#Document description for original title
data_dictionary.loc[
    data_dictionary["column_name"] == "original_title",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = ["Original movie title.",
    "Movies dataset",
     "No transformation; retained from the original dataset.",
     "Some titles may be in languages other than English."
]

#Document description for director
data_dictionary.loc[
    data_dictionary["column_name"] == "director",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = ["Movie director.",
    "Credits dataset",
     "No transformation; retained from the original dataset.",
     "Some movies do not have a director's name registered."
    ]

#Document description for top cast
data_dictionary.loc[
    data_dictionary["column_name"] == "top_cast",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = ["Leading cast members.",
    "Credits dataset",
     "No transformation; retained from the original dataset.",
     "Some information about the cast is missing or restricted."
    ]

#Document description for keywords list
data_dictionary.loc[
    data_dictionary["column_name"] == "keywords_list",
    ["description", "source", "transformation_or_derivation", "notes_or_concerns"]
] = ["List of words that can be associated with a movie.",
    "Keywords dataset",
    "No transformation; retained from the original dataset.",
    "Some movies have keywords that are missing"
    ]

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head(11)


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,movieId,int64,Unique identifier per movie.,Movies dataset,No transformation; retained from the original ...,Used this as a key to join audience rating CTE...
1,user_rating_mean,float64,Mean audience rating per movie.,Ratings dataset,"Calculated using AVG(rating), grouped by movieId",Used mean values of 100 or more as a parameter...
2,user_rating_count,int64,Number of audience ratings per movie.,Ratings dataset,"Calculated using COUNT(rating), grouped by mov...",Number of ratings per movie. Some movies have ...
3,popularity,float64,Popularity score of the movie.,Movies dataset,No transformation; retained from the original ...,Popularity scores may differ based on the movie.
4,budget,float64,Reported production budget.,Movies dataset,No transformation; retained from the original ...,Some of the budget entries contain zeros. This...
5,revenue,float64,Reported movie revenue.,Movies dataset,No transformation; retained from the original ...,Some of the revenue entries contain zeros. Thi...
6,tmdbId,int64,Unique identifier in The Movie Database,Movies & Credits datasets,No transformation; retained from the original ...,Used to connect the Movies dataset to the Cred...
7,original_title,str,Original movie title.,Movies dataset,No transformation; retained from the original ...,Some titles may be in languages other than Eng...
8,director,str,Movie director.,Credits dataset,No transformation; retained from the original ...,Some movies do not have a director's name regi...
9,top_cast,str,Leading cast members.,Credits dataset,No transformation; retained from the original ...,Some information about the cast is missing or ...


### Preliminary ABT, Validation, and Reproducible Submission

In [160]:
print("ABT dimensions:", movies_abt.shape)

print("Unique movie IDs:", movies_abt["movieId"].nunique())

print("Duplicate movie IDs:", movies_abt["movieId"].duplicated().sum())

display(movies_abt.isna().sum())

ABT dimensions: (9025, 11)
Unique movie IDs: 9025
Duplicate movie IDs: 0


movieId                 0
user_rating_mean        0
user_rating_count       0
popularity              0
budget               4409
revenue              4288
tmdbId                  0
original_title          0
director               24
top_cast               90
keywords_list         755
dtype: int64

In [161]:
abt_validation = pd.Series({
    "movie_rows_before_merges": len(movies),
    "abt_rows_after_merges": len(movies_abt),
    "duplicate_movie_ids": movies_abt["movieId"].duplicated().sum(),
    "duplicate_tmdb_ids": movies_abt["tmdbId"].duplicated().sum(),
    "movies_with_rating_summary": movies_abt["user_rating_count"].notna().sum(),
    "movies_eligible_for_rating_target": movies_abt["user_rating_mean"].notna().sum(),
})

display(abt_validation)
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


movie_rows_before_merges             9082
abt_rows_after_merges                9025
duplicate_movie_ids                     0
duplicate_tmdb_ids                      0
movies_with_rating_summary           9025
movies_eligible_for_rating_target    9025
dtype: int64

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,user_rating_mean,user_rating_count,user_rating_std,first_rating_date,last_rating_date,director,top_cast,keywords_list,has_revenue,rating_target_eligible,high_user_rating
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995,3.872470,247.0,0.958981,1996-03-30 19:00:13,2016-10-06 19:55:11,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,jealousy|toy|boy|friendship|friends|rivalry|bo...,1,True,1
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995,3.401869,107.0,0.880714,1996-03-30 19:12:30,2016-08-01 17:42:33,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst,board game|disappearance|based on children's b...,1,True,0
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,3.161017,59.0,1.150115,1996-06-05 06:19:04,2016-08-16 22:07:21,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,fishing|best friend|duringcreditsstinger|old men,0,True,0
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995,2.384615,13.0,0.938835,1996-06-10 16:45:35,2004-07-27 06:14:12,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,based on novel|interracial relationship|single...,1,False,<NA>
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,0,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995,3.267857,56.0,0.948512,1996-04-14 14:23:59,2016-08-16 22:15:47,Charles Shyer,Steve Martin|Diane Keaton|Martin Short,baby|midlife crisis|confidence|aging|daughter|...,1,True,0


,missing_rate
high_user_rating,0.856860
user_rating_std,0.341224
keywords_list,0.083792
top_cast,0.009910
last_rating_date,0.006276
user_rating_count,0.006276
user_rating_mean,0.006276
first_rating_date,0.006276
genres_list,0.003854
primary_genre,0.003854


Use this cell to document the ABT design required by the Canvas Milestone 2 page.

### Milestone 2: Preliminary Analytical Base Table

The preliminary ABT combines the Movies, Ratings, Credits, and Keywords datasets. The unit of analysis is one movie, identified by `movieId`.

The Ratings dataset is aggregated by `movieId` to calculate the average audience rating (`avg_rating`) and the number of ratings (`rating_count`). The aggregated ratings are then joined to Movies using `movieId`. Credits and Keywords are incorporated using `tmdbId`.

The resulting ABT includes movie characteristics such as popularity, budget, revenue, original title, director, cast, and keywords. These variables allow audience ratings to be compared across different movie characteristics.

The average audience rating is the primary outcome of interest. The rating count provides context for interpreting the reliability of each movie's average rating.

Zero values in budget and revenue are retained because they have not been confirmed as invalid values. Missing information in Credits and Keywords is also retained rather than automatically removing those movies.

## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [162]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [163]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
movieId,9082.0,NaN,NaN,NaN,30963.741577,1.0,2843.25,6265.5,56026.25,163949.0,40659.690679
tmdbId,9082.0,NaN,NaN,NaN,38710.463885,2.0,9446.25,15775.0,39010.25,416437.0,62123.411079
imdb_id,9082,9082,tt0114709,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,9082,8809,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_title,9082,8841,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_language,9082,42,en,7947,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_date,9082,6002,1994-01-01,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
runtime,9082.0,NaN,NaN,NaN,105.661418,0.0,93.0,102.0,115.0,1140.0,30.350712
budget,9082.0,NaN,NaN,NaN,16647890.737833,0.0,0.0,100000.0,20000000.0,380000000.0,33452764.781378
revenue,9082.0,NaN,NaN,NaN,49187276.79586,0.0,0.0,271608.0,36725282.5,2787965087.0,130180082.298936


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [164]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
# report_path = REPORTS_DIR / "movies_final_report.html"


In [165]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
